In [ ]:
# Google Colab cell for cloud API

# Install dependencies
!pip install fastapi uvicorn unsloth sentence-transformers joblib torch spacy transformers huggingface-hub pydantic numpy pyngrok
! pip install huggingface_hub[hf_xet]

# Install spaCy model
!python -m spacy download en_core_web_sm

# Install ngrok
# !pip install pyngrok

# curl -X POST "https://1c42-34-23-153-154.ngrok-free.app/grammar_correction" \
# -H "Content-Type: application/json" \
# -d '{
#     "user_id": 123,
#     "sentence": "He go to school"
# }'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.3/264.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.1/132.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Optional
import uvicorn
from unsloth import FastLanguageModel
from peft import PeftModel
import torch
import re
import traceback
from huggingface_hub import snapshot_download
from pyngrok import ngrok
import nest_asyncio
import threading
import asyncio
import time

# Allow nested event loops (for environments like Colab)
nest_asyncio.apply()

# Define FastAPI app
app = FastAPI(title="Grammar Correction API")

# Model paths and configurations
BASE_MODEL = "unsloth/mistral-7b-v0.3-bnb-4bit"
ADAPTER = "Justin73/grammar-correction-model-combined-mistral"
model, tokenizer = None, None

# Define request model
class GrammarCorrectionRequest(BaseModel):
    user_id: int
    sentence: str

# Define response model
class GrammarCorrectionResponse(BaseModel):
    success: bool
    user_id: int
    original: str
    corrected: str
    corrections: List[str]  # Simplified to just strings

def load_model():
    global model, tokenizer
    try:
        print("Loading model...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            BASE_MODEL,
            max_seq_length=2048,
            load_in_4bit=True,
        )
        model = PeftModel.from_pretrained(model, ADAPTER)
        tokenizer.pad_token = tokenizer.eos_token
        model = FastLanguageModel.for_inference(model)
        print("Model loaded!")
        return True
    except Exception as e:
        print(f"Model loading failed: {str(e)}")
        return False

@app.on_event("startup")
async def startup_event():
    """Initialize the model when the API starts"""
    load_model()

@app.post("/grammar_correction", response_model=GrammarCorrectionResponse)
async def grammar_correction(request: GrammarCorrectionRequest):
    if model is None or tokenizer is None:
        raise HTTPException(status_code=503, detail="Model not loaded")

    try:
        prompt = f"""### Instruction:
Correct all the grammatical errors in the following sentence.

### Input:
{request.sentence}

### Response:
"""

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            temperature=0.1,
            pad_token_id=tokenizer.pad_token_id
        )

        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Parse corrected text
        corrected_match = re.search(r'Corrected text:(.*?)(\n|$)', full_output, re.DOTALL)
        corrected_text = corrected_match.group(1).strip() if corrected_match else request.sentence

        # Parse corrections
        corrections = []
        corrections_section = re.search(r'Corrections:(.*?)(\n\n|$)', full_output, re.DOTALL)
        if corrections_section:
            corrections_text = corrections_section.group(1).strip()
            corrections = [
                line.strip()
                for line in corrections_text.split('\n')
                if line.strip() and re.match(r'^\d+\.', line)
            ]

        return {
            "success": True,
            "user_id": request.user_id,
            "original": request.sentence,
            "corrected": corrected_text,
            "corrections": corrections
        }

    except Exception as e:
        return {
            "success": False,
            "user_id": request.user_id,
            "original": request.sentence,
            "corrected": request.sentence,
            "corrections": []
        }

# Server functions
def run_server(port: int = 8000, max_attempts: int = 5):
    """Run the FastAPI server"""
    import socket
    for attempt in range(max_attempts):
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
                s.bind(("0.0.0.0", port))

            config = uvicorn.Config(
                app,
                host="0.0.0.0",
                port=port,
                log_level="info",
                timeout_keep_alive=30
            )
            server = uvicorn.Server(config)
            print(f"Starting server on port {port}...")
            asyncio.run(server.serve())
            break
        except OSError as e:
            if "address already in use" in str(e).lower() and attempt < max_attempts - 1:
                print(f"Port {port} in use, trying port {port + 1}...")
                port += 1
                time.sleep(1)
            else:
                print(f"Failed to start server: {str(e)}")
                raise
        except Exception as e:
            print(f"Server error: {str(e)}")
            raise

def start_ngrok(port: int, auth_token: str = None) -> Optional[str]:
    """Start an ngrok tunnel to the server"""
    try:
        # Kill any existing ngrok processes
        ngrok.kill()

        # Set auth token if provided
        if auth_token:
            ngrok.set_auth_token(auth_token)

        # Connect to ngrok
        public_url = ngrok.connect(port, bind_tls=True).public_url
        print(f"Ngrok tunnel created: {public_url}")
        return public_url
    except Exception as e:
        print(f"Ngrok error: {str(e)}")
        print("Continuing without ngrok - you'll need to access the API locally")
        return None

def start_application():
    """Start the complete application with server and ngrok"""
    port = 8000
    max_attempts = 5
    attempt = 0

    # Replace with your ngrok auth token
    ngrok_auth_token = "2wtb6WR3xstGZZcH4TKlqRXF2XR_59nwNBdA3jrCDBvrhjVW4"  # Set to None if not using ngrok

    while attempt < max_attempts:
        try:
            # Start server in a separate thread
            server_thread = threading.Thread(target=run_server, args=(port,))
            server_thread.daemon = True
            server_thread.start()

            # Wait for server to start
            time.sleep(2)

            # Start ngrok if auth token is provided
            public_url = None
            if ngrok_auth_token:
                public_url = start_ngrok(port, ngrok_auth_token)

            if public_url:
                print(f"\nAPI is available at: {public_url}/grammar_correction")
                print(f"Swagger UI: {public_url}/docs")
            else:
                print(f"\nAPI is running locally at: http://localhost:{port}/grammar_correction")
                print(f"Swagger UI: http://localhost:{port}/docs")

            # Keep the main thread alive
            while True:
                time.sleep(1)

        except KeyboardInterrupt:
            print("\nShutting down server...")
            if ngrok_auth_token:
                ngrok.kill()
            import os
            os._exit(0)
        except Exception as e:
            print(f"Application error: {str(e)}")
            if ngrok_auth_token:
                ngrok.kill()
            attempt += 1
            port += 1
            time.sleep(2)
            if attempt < max_attempts:
                print(f"Retrying with port {port}...")

# Example usage
if __name__ == "__main__":
    # Option 1: Just run the server locally
    # uvicorn.run(app, host="0.0.0.0", port=8000)

    # Option 2: Run with ngrok for public access
    start_application()